# Лабораторная работа №4

Многокритериальная оптимизация параметров модели SIR

Закиров Нурислам Дамирович (РУДН)

# Введение

В данной работе проводится многокритериальная оптимизация параметров
модели SIR для поиска компромиссных стратегий управления эпидемией.

## Цели оптимизации

Минимизация одновременно двух критериев:

1.  **Пиковая заболеваемость** — нагрузка на систему здравоохранения
2.  **Доля умерших** — демографические потери

## Оптимизируемые параметры

-   **β_und** — в диапазоне \[0.1, 1.0\]
-   **detection_time** — в диапазоне \[3, 14\] (целые)
-   **death_rate** — в диапазоне \[0.01, 0.1\]

# Инициализация проекта

In [1]:
using DrWatson
@quickactivate "project"
using BlackBoxOptim, Random, Statistics
include(srcdir("sir_model.jl"))

total_count (generic function with 1 method)

# Целевая функция

In [2]:
# Целевая функция: минимизируем пиковую заболеваемость и смертность
function cost_multi(x)
    # x[1]: β_und, x[2]: death_rate, x[3]: detection_time
    model = initialize_sir(;
        Ns = [1000, 1000, 1000],
        β_und = fill(x[1], 3),
        β_det = fill(x[1]/10, 3),
        infection_period = 14,
        detection_time = round(Int, x[2]),
        death_rate = x[3],
        reinfection_probability = 0.1,
        Is = [0, 0, 1],
        seed = 42,
        n_steps = 100,
    )

    infected_frac(model) = count(a.status == :I for a in allagents(model)) / nagents(model)
    dead_count(model) = 3000 - nagents(model)

    peak_infected = 0.0

    # Запускаем с несколькими повторами
    replicates = 5
    peak_vals = Float64[]
    dead_vals = Int[]

    for rep = 1:replicates
        model = initialize_sir(;
            Ns = [1000, 1000, 1000],
            β_und = fill(x[1], 3),
            β_det = fill(x[1]/10, 3),
            infection_period = 14,
            detection_time = round(Int, x[2]),
            death_rate = x[3],
            reinfection_probability = 0.1,
            Is = [0, 0, 1],
            seed = 42 + rep,
            n_steps = 100,
        )

        for step = 1:100
            Agents.step!(model, 1)
            frac = infected_frac(model)
            if frac > peak_infected
                peak_infected = frac
            end
        end
        push!(peak_vals, peak_infected)
        push!(dead_vals, dead_count(model))
    end

    return (mean(peak_vals), mean(dead_vals) / 3000)  # доля умерших
end

cost_multi (generic function with 1 method)

# Запуск оптимизации

In [3]:
# Запуск оптимизации
result = bboptimize(
    cost_multi,
    Method = :borg_moea,
    FitnessScheme = ParetoFitnessScheme{2}(is_minimizing = true),
    SearchRange = [
        (0.1, 1.0),    # β_und
        (3.0, 14.0),   # detection_time
        (0.01, 0.1),   # death_rate
    ],
    NumDimensions = 3,
    MaxTime = 120,  # 2 минуты
    TraceMode = :compact,
)

best = best_candidate(result)
fitness = best_fitness(result)

Starting optimization with optimizer BlackBoxOptim.BorgMOEA{EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}, BlackBoxOptim.ProblemEvaluator{Tuple{Float64, Float64}, IndexedTupleFitness{2, Float64}, EpsBoxArchive{2, Float64, EpsBoxDominanceFitnessScheme{2, Float64, true, typeof(sum)}}, FunctionBasedProblem{typeof(cost_multi), ParetoFitnessScheme{2, Float64, true, typeof(sum)}, ContinuousRectSearchSpace, Nothing}}, FitPopulation{IndexedTupleFitness{2, Float64}}, FixedGeneticOperatorsMixture, RandomBound{ContinuousRectSearchSpace}}
0.00 secs, 0 evals, 0 steps
pop.size=50 arch.size=0 n.restarts=0

Optimization stopped after 1 steps and 1299.60 seconds
Termination reason: Max time (120.0 s) reached
Steps per second = 0.00
Function evals per second = 0.04
Improvements/step = NaN
Total function evaluations = 51


Best candidate found: [0.156209, 3.38514, 0.0629002]

Fitness: (0.00067, 0.00000) agg=0.00067


(0.0006666666666666666, 0.0)

# Результаты оптимизации

In [4]:
println("Оптимальные параметры:")
println("β_und = $(best[1])")
println("Время выявления = $(round(Int, best[2])) дней")
println("Смертность = $(best[3])")
println("Достигнутые показатели:")
println("Пик заболеваемости: $(fitness[1])")
println("Доля умерших: $(fitness[2])")

save(datadir("optimization_result.jld2"), Dict("best" => best, "fitness" => fitness))

Оптимальные параметры:
β_und = 0.15620863503082144
Время выявления = 3 дней
Смертность = 0.06290015281515267
Достигнутые показатели:
Пик заболеваемости: 0.0006666666666666666
Доля умерших: 0.0

# Анализ результатов

Оптимизация позволяет найти **Парето-фронт** — набор компромиссных
решений, где:

-   Улучшение одного критерия ухудшает другой
-   Можно выбрать стратегию в зависимости от приоритетов

## Практическое значение

1.  **Раннее выявление** (малое detection_time) снижает пик
    заболеваемости
2.  **Снижение смертности** требует дополнительных медицинских ресурсов
3.  **Компромисс** между экономическими затратами и сохранением жизней

# Выводы

Многокритериальная оптимизация позволила:

1.  Найти набор Парето-оптимальных стратегий управления
2.  Оценить компромисс между пиковой нагрузкой и смертностью
3.  Обосновать выбор параметров для различных сценариев

# Литература

1.  Hethcote H. W. The Mathematics of Infectious Diseases. — 2000.
2.  BlackBoxOptim.jl: Black-Box Optimization for Julia. — 2024.